<a href="https://colab.research.google.com/github/shivashankarb2006/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivashankarb2006/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Method Choice and Why

I chose Random Forest Classification for the Content Opportunity Scoring lane.

The task is to identify content pages that are potential optimization opportunities. Random Forest can combine several search-performance and content signals and capture non-linear relationships between them.

It is also useful for interpretation because feature importance can show which signals contribute most to the model's decisions.

The model is used for decision support and ranking, not as proof of causation or as a prediction of Google's ranking algorithm.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Split Design

I use a GroupShuffleSplit based on client_hash_id.

Pages from the same client are kept in the same train or test group. This reduces the risk that the model learns client-specific patterns and then appears to perform well simply because similar pages from the same client occur in both sets.

The split is therefore intended to provide a more honest estimate of whether the model can generalize across clients.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [19]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

In [20]:
%pip -q install duckdb huggingface_hub scikit-learn pandas

In [21]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("DuckDB connected successfully.")

DuckDB connected successfully.


In [22]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions ELSE 0
                END
            ) AS imp_last30,

            SUM(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions ELSE 0
                END
            ) AS imp_prev30,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_clicks ELSE 0
                END
            ) AS clk_last30,

            AVG(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_prev30

        FROM {TABLES['fact_daily']} f, bounds b

        WHERE f.report_date > b.end_d - INTERVAL 90 DAY

        GROUP BY 1, 2

        HAVING imp_prev30 >= 50
    )

    SELECT *
    FROM windowed
""").df()

print("Feature rows:", len(features))
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 155903


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_prev30
0,client_3ffa76342f366962,content_b89167cd03d6ffc1,12.0,102.0,1.0,4.620238
1,client_3ffa76342f366962,content_f33fd2cd160185e7,41.0,130.0,0.0,6.293744
2,client_e547b89c05043229,content_2e296120acb03e93,2346.0,5207.0,0.0,46.106395
3,client_e547b89c05043229,content_516b7c0e8eec0cef,371.0,2494.0,0.0,47.851813
4,client_e547b89c05043229,content_38b6c1a9aa29f801,5746.0,12657.0,2.0,42.586554


In [23]:
qsignals = con.sql(f"""
    SELECT
        content_hash_id,

        ANY_VALUE(content_visible_query_count)
            AS visible_queries,

        ANY_VALUE(rare_impressions_share)
            AS rare_share,

        ANY_VALUE(anonymized_impressions_share)
            AS anon_share,

        MAX(impressions_90d)
            AS top_query_impressions,

        SUM(impressions_90d)
            AS kept_impressions

    FROM {TABLES['fact_query_90d']}

    GROUP BY content_hash_id
""").df()

qsignals["top_query_share"] = (
    qsignals["top_query_impressions"] /
    qsignals["kept_impressions"]
)

qsignals["query_diversity"] = (
    1 - qsignals["top_query_share"]
)

data = features.merge(
    qsignals,
    on="content_hash_id",
    how="left"
)

print("Modeling rows:", len(data))
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 155903


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_prev30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share,query_diversity
0,client_3ffa76342f366962,content_b89167cd03d6ffc1,12.0,102.0,1.0,4.620238,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,client_3ffa76342f366962,content_f33fd2cd160185e7,41.0,130.0,0.0,6.293744,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,client_e547b89c05043229,content_2e296120acb03e93,2346.0,5207.0,0.0,46.106395,43.0,0.042632,0.545876,409.0,3108.0,0.131596,0.868404
3,client_e547b89c05043229,content_516b7c0e8eec0cef,371.0,2494.0,0.0,47.851813,13.0,0.112391,0.679232,305.0,597.0,0.510888,0.489112
4,client_e547b89c05043229,content_38b6c1a9aa29f801,5746.0,12657.0,2.0,42.586554,129.0,0.069119,0.517198,1398.0,7613.0,0.183633,0.816367


In [24]:
data["is_declining"] = (
    data["imp_last30"] < 0.8 * data["imp_prev30"]
).astype(int)

feature_cols = [
    "imp_prev30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share",
    "query_diversity"
]

model_data = data.dropna(
    subset=feature_cols + ["client_hash_id"]
).copy()

X = model_data[feature_cols]
y = model_data["is_declining"]
groups = model_data["client_hash_id"]

print("Model rows:", len(model_data))
print("Declining rate:", y.mean())

Model rows: 124268
Declining rate: 0.8673190201821869


In [25]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

Train rows: 108877
Test rows: 15391


In [26]:
baseline_label = y_train.mode().iloc[0]

baseline_pred = [baseline_label] * len(y_test)

baseline_metrics = {
    "accuracy": accuracy_score(y_test, baseline_pred),
    "precision": precision_score(
        y_test, baseline_pred, zero_division=0
    ),
    "recall": recall_score(
        y_test, baseline_pred, zero_division=0
    ),
    "f1": f1_score(
        y_test, baseline_pred, zero_division=0
    )
}

baseline_metrics

{'accuracy': 0.8546553180430122,
 'precision': 0.8546553180430122,
 'recall': 1.0,
 'f1': 0.9216325100718165}

In [27]:
model_metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(
        y_test, pred, zero_division=0
    ),
    "recall": recall_score(
        y_test, pred, zero_division=0
    ),
    "f1": f1_score(
        y_test, pred, zero_division=0
    )
}

model_metrics

{'accuracy': 0.9291793905529205,
 'precision': 0.9273770724103727,
 'recall': 0.9950585373270489,
 'f1': 0.9600264045767933}

In [28]:
comparison = pd.DataFrame(
    [baseline_metrics, model_metrics],
    index=["Baseline", "Random Forest"]
)

comparison

,accuracy,precision,recall,f1
Baseline,0.854655,0.854655,1.000000,0.921633
Random Forest,0.929179,0.927377,0.995059,0.960026


In [29]:
import pandas as pd

## Errors and Interpretation

The model should not be judged only by its overall accuracy.

I will inspect false positives and false negatives to understand where the model makes mistakes.

A false positive means that the model identifies a page as declining when the observed label does not indicate a decline.

A false negative means that the model misses a page that meets the decline definition.

Feature importance will be used as a directional interpretation of which signals the model relies on. Feature importance does not establish causality.

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [31]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

importance

,feature,importance
3,anon_share,0.231402
0,imp_prev30,0.229049
2,rare_share,0.188941
1,visible_queries,0.129625
5,query_diversity,0.110916
4,top_query_share,0.110067


In [32]:
errors = model_data.iloc[test_idx].copy()

errors["actual"] = y_test.values
errors["predicted"] = pred

print("False positives:")
display(
    errors[
        (errors["actual"] == 0) &
        (errors["predicted"] == 1)
    ][feature_cols + ["actual", "predicted"]].head(10)
)

print("False negatives:")
display(
    errors[
        (errors["actual"] == 1) &
        (errors["predicted"] == 0)
    ][feature_cols + ["actual", "predicted"]].head(10)
)

False positives:


,imp_prev30,visible_queries,rare_share,anon_share,top_query_share,query_diversity,actual,predicted
15388,195.0,2.0,0.294906,0.640751,0.541667,0.458333,0,1
15534,253.0,3.0,0.327044,0.559748,0.537037,0.462963,0,1
15689,3333.0,25.0,0.120350,0.692210,0.173028,0.826972,0,1
15889,909.0,25.0,0.264142,0.184666,0.103478,0.896522,0,1
15957,1846.0,15.0,0.013292,0.734541,0.239878,0.760122,0,1
16154,168.0,1.0,0.473684,0.490132,1.000000,0.000000,0,1
16578,1133.0,3.0,0.011152,0.926064,0.750000,0.250000,0,1
16579,1400.0,6.0,0.006486,0.954304,0.255639,0.744361,0,1
16854,11376.0,46.0,0.008477,0.716878,0.356910,0.643090,0,1
16869,334.0,9.0,0.286834,0.344828,0.242553,0.757447,0,1


False negatives:


,imp_prev30,visible_queries,rare_share,anon_share,top_query_share,query_diversity,actual,predicted
16426,2453.0,11.0,0.024907,0.500505,0.827660,0.172340,1,0
17148,83.0,1.0,0.047059,0.776471,1.000000,0.000000,1,0
17375,594.0,4.0,0.046243,0.893642,0.346154,0.653846,1,0
18118,878.0,35.0,0.444937,0.085735,0.102362,0.897638,1,0
18169,104.0,5.0,0.467949,0.153846,0.288136,0.711864,1,0
18210,611.0,6.0,0.036503,0.865514,0.294118,0.705882,1,0
18227,51.0,3.0,0.253521,0.253521,0.400000,0.600000,1,0
48231,132.0,2.0,0.253589,0.315789,0.877778,0.122222,1,0
48798,1327.0,12.0,0.021513,0.529493,0.462133,0.537867,1,0
59092,259.0,1.0,0.166282,0.810624,1.000000,0.000000,1,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.